# 34 GPU VLM mechanics

L4/A100 推奨。VLM manifest template、HF open-source VLM caption/tags/scores、VLM feature baseline をまとめます。

各 stage は `BASE_DIR/reports/preflight/compact_runs/` に JSONL log を追記します。既に期待 artifact が揃っている stage は skip します。再実行したい場合は `FORCE_RERUN_ALL=True` にしてください。

In [ ]:
from pathlib import Path
import importlib.util
import os
import sys


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


# v2 をデフォルトにします。v1 を再実行したい時だけここを書き換えてください。
os.environ.setdefault('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
os.environ.setdefault('BATTING_CODE_ROOT', '/content/drive/MyDrive/codex/batting_codex_handoff')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path(os.environ['BATTING_CODE_ROOT'])
    mydrive = Path('/content/drive/MyDrive')
    candidates = [
        fixed,
        Path('/content/drive/MyDrive/codex/batting_codex_handoff'),
        Path('/content/drive/MyDrive/batting_codex_handoff'),
        Path.cwd(),
        *Path.cwd().parents,
    ]
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            return candidate
    raise FileNotFoundError(
        'sport_pipeline repo が見つかりません。Drive mount と repo 配置を確認してください。\n'
        f'BATTING_CODE_ROOT={fixed}\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別配置の場合は compact notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/あなたの配置/batting_codex_handoff\n'
        f'MyDrive exists={mydrive.exists()}\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()
REPO_DIR = _resolve_repo_dir()
os.environ['BATTING_CODE_ROOT'] = str(REPO_DIR)
NOTEBOOK_DIR = REPO_DIR / 'notebooks'
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ['BASEBALL_VISION_RUN_PROFILE']
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME
src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(f'sport_pipeline を import できません: {src_dir}')

from sport_pipeline.compact_notebooks import CompactRunLogger
from sport_pipeline.io import read_table
from sport_pipeline.pipeline.run_profile import artifact_id, load_run_profile, run_id, stage_settings

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
FULL_RUN_ID = run_id(RUN_PROFILE, 'full_run_id')
CONTEXT_RUN_ID = run_id(RUN_PROFILE, 'context_run_id')
SEQUENCE_RUN_ID = run_id(RUN_PROFILE, 'sequence_run_id')
SEQUENCE_TCN_RUN_ID = run_id(RUN_PROFILE, 'sequence_tcn_run_id')
VIDEO_LIGHTWEIGHT_RUN_ID = run_id(RUN_PROFILE, 'video_lightweight_run_id')
VIDEO_FROZEN_RUN_ID = run_id(RUN_PROFILE, 'video_frozen_run_id')
VIDEO_FINETUNE_RUN_ID = run_id(RUN_PROFILE, 'video_finetune_run_id')
PLAYER_SEASON_RUN_ID = run_id(RUN_PROFILE, 'player_season_run_id')
VLM_RUN_ID = run_id(RUN_PROFILE, 'vlm_run_id')
FUSION_RUN_ID = run_id(RUN_PROFILE, 'fusion_run_id')
METHOD_EVALUATION_REPORT_ID = run_id(RUN_PROFILE, 'method_evaluation_report_id')
VIDEO_ABLATION_REPORT_ID = run_id(RUN_PROFILE, 'video_ablation_report_id')
STRUCTURED_SEQUENCE_FEATURE_ID = artifact_id(RUN_PROFILE, 'structured_sequence_feature_id')
CLIP_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'clip_embedding_feature_id')
PLAYER_SEASON_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'player_season_embedding_feature_id')
VIDEO_LIGHTWEIGHT_FEATURE_ID = artifact_id(RUN_PROFILE, 'video_lightweight_feature_id', 'video_lightweight_features_mlb_2024_2026_v2')
VIDEO_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'video_embedding_feature_id')
VLM_FEATURE_ID = artifact_id(RUN_PROFILE, 'vlm_feature_id')

print('RUN_PROFILE =', RUN_PROFILE_NAME)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('BASE_DIR =', BASE_DIR)
print('NOTEBOOK_DIR =', NOTEBOOK_DIR)
print('src_dir =', src_dir)


In [ ]:
import json

COMPACT_RUN_NAME = '34_gpu_vlm_mechanics'
LOGGER = CompactRunLogger(BASE_DIR, COMPACT_RUN_NAME, run_profile_name=RUN_PROFILE_NAME)
FORCE_RERUN_ALL = False

VLM_SETTINGS = stage_settings(RUN_PROFILE, 'vlm_mechanics')
VLM_EXECUTE_DEFAULT = bool(VLM_SETTINGS.get('execute_default', False))

def _env_bool(name, default):
    value = os.environ.get(name)
    return default if value is None else value.lower() in {'1', 'true', 'yes'}

INSTALL_HF_VLM_DEPS = _env_bool('INSTALL_HF_VLM_DEPS', bool(VLM_SETTINGS.get('install_hf_vlm_deps_default', VLM_EXECUTE_DEFAULT)))
RUN_HF_VLM_CAPTIONING = _env_bool('RUN_HF_VLM_CAPTIONING', bool(VLM_SETTINGS.get('run_hf_captioning_default', VLM_EXECUTE_DEFAULT)))
RUN_VLM_BASELINE = _env_bool('RUN_VLM_BASELINE', bool(VLM_SETTINGS.get('run_baseline_default', VLM_EXECUTE_DEFAULT)))
HF_VLM_MAX_ROWS = int(os.environ.get('HF_VLM_MAX_ROWS') or VLM_SETTINGS.get('caption_max_rows', 100))

VLM_MANIFEST_PATH = BASE_DIR / 'features' / VLM_FEATURE_ID / 'manifest.parquet'
VLM_CAPTION_SUMMARY_PATH = BASE_DIR / 'reports' / 'preflight' / f'hf_vlm_captioning_{VLM_FEATURE_ID}.json'

def _safe_vlm_rows(path):
    try:
        return read_table(path) if path.exists() else []
    except Exception as exc:
        print('WARNING: VLM manifest check failed:', path, exc)
        return []

def _vlm_complete_count(rows):
    return sum(1 for row in rows if row.get('vlm_caption') or row.get('feature_status') == 'vlm_complete')

def _vlm_status_counts(rows):
    counts = {}
    for row in rows:
        status = str(row.get('feature_status') or 'missing')
        counts[status] = counts.get(status, 0) + 1
    return counts

def _caption_failure_preview(path, rows):
    failures = []
    if path.exists():
        try:
            payload = json.loads(path.read_text(encoding='utf-8'))
            failures.extend(payload.get('failures') or [])
        except Exception as exc:
            failures.append({'summary_read_error': str(exc)})
    if not failures:
        for row in rows:
            if row.get('feature_status') == 'vlm_failed':
                failures.append({
                    'clip_id': row.get('clip_id'),
                    'vlm_error': row.get('vlm_error'),
                    'vlm_video_error': row.get('vlm_video_error'),
                })
            if len(failures) >= 5:
                break
    return failures[:5]

vlm_rows_before = _safe_vlm_rows(VLM_MANIFEST_PATH)
vlm_complete_rows_before = _vlm_complete_count(vlm_rows_before)
FORCE_REBUILD_VLM_TEMPLATE = RUN_HF_VLM_CAPTIONING and len(vlm_rows_before) < HF_VLM_MAX_ROWS
FORCE_VLM_CAPTIONING_FOR_TARGET = RUN_HF_VLM_CAPTIONING and vlm_complete_rows_before < HF_VLM_MAX_ROWS
FORCE_VLM_BASELINE_FOR_TARGET = RUN_VLM_BASELINE and (FORCE_VLM_CAPTIONING_FOR_TARGET or FORCE_RERUN_ALL)

os.environ['FORCE_REBUILD_VLM_TEMPLATE'] = '1' if FORCE_REBUILD_VLM_TEMPLATE else '0'
os.environ['INSTALL_HF_VLM_DEPS'] = '1' if INSTALL_HF_VLM_DEPS else '0'
os.environ['RUN_HF_VLM_CAPTIONING'] = '1' if RUN_HF_VLM_CAPTIONING else '0'
os.environ['HF_VLM_MAX_ROWS'] = str(HF_VLM_MAX_ROWS)

print('VLM execute_default =', VLM_EXECUTE_DEFAULT)
print('INSTALL_HF_VLM_DEPS =', INSTALL_HF_VLM_DEPS)
print('RUN_HF_VLM_CAPTIONING =', RUN_HF_VLM_CAPTIONING)
print('RUN_VLM_BASELINE =', RUN_VLM_BASELINE)
print('HF_VLM_MAX_ROWS target complete rows =', HF_VLM_MAX_ROWS)
print('vlm_rows_before =', len(vlm_rows_before), 'complete =', vlm_complete_rows_before, 'status_counts =', _vlm_status_counts(vlm_rows_before))
print('force_rebuild_vlm_template =', FORCE_REBUILD_VLM_TEMPLATE)
print('force_vlm_captioning_for_target =', FORCE_VLM_CAPTIONING_FOR_TARGET)

# 1回目の 24 は template 作成だけ。
os.environ['RUN_VLM_BASELINE'] = '0'
LOGGER.run_stage(
    ipython=get_ipython(), notebook_dir=NOTEBOOK_DIR, index=1, total=3,
    notebook='24_vlm_mechanics_baseline.ipynb', label='Create/keep VLM mechanics template',
    force=FORCE_RERUN_ALL or FORCE_REBUILD_VLM_TEMPLATE,
    expected_outputs=[f'features/{VLM_FEATURE_ID}/manifest.parquet'],
    progress_files=[f'reports/preflight/vlm_feature_template_{VLM_FEATURE_ID}.json'],
)
LOGGER.run_stage(
    ipython=get_ipython(), notebook_dir=NOTEBOOK_DIR, index=2, total=3,
    notebook='24b_hf_vlm_captioning.ipynb', label='HF VLM caption/tags/scores',
    enabled=RUN_HF_VLM_CAPTIONING, force=FORCE_VLM_CAPTIONING_FOR_TARGET or FORCE_RERUN_ALL,
    expected_outputs=[] if RUN_HF_VLM_CAPTIONING else [f'features/{VLM_FEATURE_ID}/manifest.parquet'],
    progress_files=[f'reports/preflight/hf_vlm_captioning_{VLM_FEATURE_ID}.json'],
)

vlm_rows_after = _safe_vlm_rows(VLM_MANIFEST_PATH)
vlm_complete_rows_after = _vlm_complete_count(vlm_rows_after)
vlm_failed_rows_after = sum(1 for row in vlm_rows_after if row.get('feature_status') == 'vlm_failed')
fallback_rows_after = sum(1 for row in vlm_rows_after if row.get('vlm_input_mode') == 'debug_frame_fallback')
print('vlm_rows_after =', len(vlm_rows_after), 'complete =', vlm_complete_rows_after, 'failed =', vlm_failed_rows_after, 'debug_frame_fallback =', fallback_rows_after, 'status_counts =', _vlm_status_counts(vlm_rows_after))
if RUN_HF_VLM_CAPTIONING and VLM_MANIFEST_PATH.exists() and vlm_complete_rows_after == 0:
    print('first VLM failures:', _caption_failure_preview(VLM_CAPTION_SUMMARY_PATH, vlm_rows_after))
    raise RuntimeError('HF VLM captioning produced 0 complete rows; baseline is skipped. Inspect hf_vlm_captioning summary / VLM manifest failure columns first.')
if RUN_HF_VLM_CAPTIONING and vlm_complete_rows_after < HF_VLM_MAX_ROWS:
    print('WARNING: VLM complete rows are below target:', vlm_complete_rows_after, '/', HF_VLM_MAX_ROWS)

# 最後に必要な時だけ VLM feature baseline を実行します。
RUN_VLM_BASELINE_EFFECTIVE = RUN_VLM_BASELINE and vlm_complete_rows_after > 0
os.environ['RUN_VLM_BASELINE'] = '1' if RUN_VLM_BASELINE_EFFECTIVE else '0'
LOGGER.run_stage(
    ipython=get_ipython(), notebook_dir=NOTEBOOK_DIR, index=3, total=3,
    notebook='24_vlm_mechanics_baseline.ipynb', label='VLM feature baseline predictions',
    enabled=RUN_VLM_BASELINE_EFFECTIVE, force=FORCE_VLM_BASELINE_FOR_TARGET,
    expected_outputs=[f'predictions/{VLM_RUN_ID}/metrics_v1.json'] if RUN_VLM_BASELINE_EFFECTIVE else [f'features/{VLM_FEATURE_ID}/manifest.parquet'],
    progress_files=[f'reports/preflight/vlm_feature_baseline_{VLM_RUN_ID}.json'],
)
LOGGER.log({'status': 'compact_run_complete', 'stages': 3})
print('compact log ->', LOGGER.log_path)
